# Residue binding (binary) preprocessing

Input data for Figure 3C

In [ ]:
import os
from pathlib import Path
import urllib.request

import pandas as pd
import plotly.express as px

In [ ]:
_cwd = Path.cwd()
REPO_ROOT = next((str(p) for p in [_cwd, *_cwd.parents] if (p / ".git").exists()), str(_cwd))

RAW_FASTA    = os.path.join(REPO_ROOT, "examples/paper/data/binding/raw/all.fasta")
RAW_METAL    = os.path.join(REPO_ROOT, "examples/paper/data/binding/raw/binding_residues_2.5_metal.txt")
RAW_SMALL    = os.path.join(REPO_ROOT, "examples/paper/data/binding/raw/binding_residues_2.5_small.txt")
RAW_NUCLEAR  = os.path.join(REPO_ROOT, "examples/paper/data/binding/raw/binding_residues_2.5_nuclear.txt")
TEST_IDS     = os.path.join(REPO_ROOT, "examples/paper/data/binding/raw/uniprot_test.txt")
OUTPUT_FASTA = os.path.join(REPO_ROOT, "examples/paper/data/binding/processed/binding_train_sequences.fasta")
OUTPUT_CSV   = os.path.join(REPO_ROOT, "examples/paper/data/binding/processed/binding_train_features.csv")

## 1. Download raw data


In [ ]:
_BASE = "https://raw.githubusercontent.com/Rostlab/bindPredict/master/data/development_set"

_DOWNLOADS = {
    RAW_FASTA:   f"{_BASE}/all.fasta",
    RAW_METAL:   f"{_BASE}/binding_residues_2.5_metal.txt",
    RAW_SMALL:   f"{_BASE}/binding_residues_2.5_small.txt",
    RAW_NUCLEAR: f"{_BASE}/binding_residues_2.5_nuclear.txt",
    TEST_IDS:    f"{_BASE}/uniprot_test.txt",
}

for dest, url in _DOWNLOADS.items():
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    if not os.path.exists(dest):
        print(f"Downloading {url} ...")
        urllib.request.urlretrieve(url, dest)
        print(f"saved to {dest}")
    else:
        print(f"Already present: {dest}")

## 2. Parse sequences

In [ ]:
proteins = {}  # OrderedDict behaviour (Python 3.7+): preserves FASTA order

with open(RAW_FASTA) as fh:
    cur_id, cur_seq = None, []
    for line in fh:
        line = line.rstrip()
        if line.startswith(">"):
            if cur_id is not None:
                proteins[cur_id] = "".join(cur_seq)
            cur_id, cur_seq = line[1:].split()[0], []
        else:
            cur_seq.append(line)
    if cur_id is not None:
        proteins[cur_id] = "".join(cur_seq)

print(f"Sequences parsed: {len(proteins)}")

## 3. Filter to training split

`all.fasta` contains 1,314 proteins (1,014 training + 300 test). `uniprot_test.txt` lists
the 300 held-out test proteins; these are excluded so the features CSV covers training only.

In [ ]:
with open(TEST_IDS) as fh:
    test_ids = {line.strip() for line in fh if line.strip()}

proteins_train = {pid: seq for pid, seq in proteins.items() if pid not in test_ids}
print(f"All proteins:      {len(proteins)}")
print(f"Test proteins:     {len(test_ids)}")
print(f"Training proteins: {len(proteins_train)}")

## 3. Parse binding annotations

Each annotation file maps a protein ID to a set of 1-indexed binding residue positions.
Proteins absent from a file receive `no_information` for that class.

In [ ]:
def _parse_binding_file(path):
    """Returns {protein_id: frozenset of 1-indexed binding positions}."""
    annot = {}
    with open(path) as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            parts = line.split("\t")
            pid = parts[0]
            positions = frozenset(int(x) for x in parts[1].split(",") if x) if len(parts) > 1 and parts[1] else frozenset()
            annot[pid] = positions
    return annot

metal_annot   = _parse_binding_file(RAW_METAL)
small_annot   = _parse_binding_file(RAW_SMALL)
nuclear_annot = _parse_binding_file(RAW_NUCLEAR)

print(f"Metal annotations:   {len(metal_annot)} proteins")
print(f"Small annotations:   {len(small_annot)} proteins")
print(f"Nuclear annotations: {len(nuclear_annot)} proteins")

## 4. Build per-residue feature matrix

In [ ]:
_BIN_EDGES  = [0, 50, 100, 150, 200, 250, 300, 350, 400, 450, 500, 550, 600]
_BIN_LABELS = ["1-50", "51-100", "101-150", "151-200", "201-250", "251-300",
               "301-350", "351-400", "401-450", "451-500", "501-550", "551-600"]

def _length_bin(n):
    for i, edge in enumerate(_BIN_EDGES[1:]):
        if n <= edge:
            return _BIN_LABELS[i]
    return f"{_BIN_EDGES[-1]+1}+"

rows = []
for pid, seq in proteins_train.items():
    seq_len = len(seq)
    length_bin = _length_bin(seq_len)
    m_annot = metal_annot.get(pid)    # None → no_information
    s_annot = small_annot.get(pid)
    n_annot = nuclear_annot.get(pid)

    for pos in range(1, seq_len + 1):
        metal_val   = ("yes" if pos in m_annot else "no") if m_annot is not None else "no_information"
        small_val   = ("yes" if pos in s_annot else "no") if s_annot is not None else "no_information"
        nuclear_val = ("yes" if pos in n_annot else "no") if n_annot is not None else "no_information"

        binding_types = [t for t, v in [("metal", metal_val), ("small", small_val), ("nuclear", nuclear_val)] if v == "yes"]
        rows.append({
            "residue_id":         f"{pid}_{pos}",
            "protein_id":         pid,
            "pos":                pos,
            "binding":            "yes" if binding_types else "no",
            "metal":              metal_val,
            "small":              small_val,
            "nuclear":            nuclear_val,
            "ligand":             ",".join(binding_types) if binding_types else "none",
            "sequence_length":    seq_len,
            "sequence_length_bin": length_bin,
        })

df = pd.DataFrame(rows)
print(f"Total residue rows: {len(df):,}")
print(f"Binding:     {(df['binding']=='yes').sum():,}")
print(f"Non-binding: {(df['binding']=='no').sum():,}")

## 5. Save outputs

In [ ]:
os.makedirs(os.path.dirname(OUTPUT_FASTA), exist_ok=True)
os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)

with open(OUTPUT_FASTA, "w") as fh:
    for pid, seq in proteins_train.items():
        fh.write(f">{pid}\n{seq}\n")
print(f"Saved {len(proteins_train)} sequences to {OUTPUT_FASTA}")

df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved {len(df):,} residue rows to {OUTPUT_CSV}")

## 6. Class distribution

In [ ]:
counts = df["binding"].value_counts().reset_index()
counts.columns = ["binding", "count"]
counts["label"] = counts["binding"].map({"yes": "Binding", "no": "Non-binding"})
print(counts[["label", "count"]].to_string(index=False))

fig = px.bar(
    counts.sort_values("count", ascending=False),
    x="label", y="count", text="count",
    title="DevSet1014 — binding class distribution",
    labels={"label": "Class", "count": "# residues"},
    template="plotly_white",
    color="label",
    color_discrete_map={"Non-binding": "#AAAAAA", "Binding": "#C0392B"},
)
fig.show()